In [1]:
# Standard Libraries
import os
import re
import random
import glob
import json
import gzip
import shutil
import logging
import copy
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
from pathlib import Path

# Data Manipulation & Visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Environment & Database
from dotenv import load_dotenv
from sqlalchemy import create_engine

# Setup Path
path = os.path.dirname(os.getcwd())
print(path)

c:\Users\sandi\Desktop\ML Working Folder\restaurant_inventory_optimzation


In [2]:
# 1. Setup connection (same as before)
load_dotenv()
connection_string = f"postgresql://{os.getenv('user')}:{os.getenv('password')}@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
engine = create_engine(connection_string)

# 2. Retrieve the tables from Supabase
print("Fetching data from Supabase...")
df_train = pd.read_sql('restaurant_train', engine)
df_meal = pd.read_sql('meal_info', engine)

# 3. Join the data
# We use 'left' to ensure we keep every single row from our training data, 
# even if a meal_id somehow didn't exist in the info table.
df_combined = pd.merge(df_train, df_meal, on='meal_id', how='left')

# 4. Check the results
print(f"New shape of data: {df_combined.shape}")
print(df_combined.head())

# Save the merged version locally for quick access later
df_combined.to_csv('../data/processed/restaurant_merged_train_data.csv', index=False)

Fetching data from Supabase...
New shape of data: (456548, 13)
        id  week  center_id  meal_id  checkout_price  base_price  \
0  1379560     1         55     1885          136.83      152.29   
1  1466964     1         55     1993          136.83      135.83   
2  1346989     1         55     2539          134.86      135.86   
3  1338232     1         55     2139          339.50      437.53   
4  1448490     1         55     2631          243.50      242.50   

   emailer_for_promotion  homepage_featured  num_orders   category cuisine  \
0                      0                  0         177  Beverages    Thai   
1                      0                  0         270  Beverages    Thai   
2                      0                  0         189  Beverages    Thai   
3                      0                  0          54  Beverages  Indian   
4                      0                  0          40  Beverages  Indian   

               dish_name                                   

In [3]:
df_combined.head()

,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders,category,cuisine,dish_name,Ingredients
0,1379560,1,55,1885,136.83,152.29,0,0,177,Beverages,Thai,Thai Iced Tea,"Strong black tea, star anise, cardamom, sweete..."
1,1466964,1,55,1993,136.83,135.83,0,0,270,Beverages,Thai,Lemongrass Ginger Tea,"Fresh lemongrass, ginger slices, honey, hot water"
2,1346989,1,55,2539,134.86,135.86,0,0,189,Beverages,Thai,Nam Manao,"Lime juice, sugar syrup, water, ice, salt"
3,1338232,1,55,2139,339.50,437.53,0,0,54,Beverages,Indian,Thandai,"Milk, sugar, almonds, cashews, melon seeds, po..."
4,1448490,1,55,2631,243.50,242.50,0,0,40,Beverages,Indian,Masala Chai,"Black tea leaves, milk, ginger, green cardamom..."
